# Temporal locality and data efficiency

Train two small encoders separately on independently sampled temporal pairs from different nonlinear sensors. Compare local transitions, isospectral scrambled transitions, and an IID temporal control. Same-state cross-sensor pairs are used only after training for calibration and evaluation.

The primary measurement is held-out linear CKA. More alignment is not assumed, and high similarity alone is not evidence that physical laws have been identified.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_URL = 'https://github.com/AlexM3020/The-Role-of-Dynamical-Structure-in-Cross-Modal-Representation-Learning.git'
REPO_NAME = 'The-Role-of-Dynamical-Structure-in-Cross-Modal-Representation-Learning'
REPO_REF = 'main'  # Replace with an immutable commit for a fixed experiment.
candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / REPO_NAME]
ROOT = next((p for p in candidates if (p / 'src/dynalign/worlds.py').exists()), None)
if ROOT is None:
    ROOT = Path.cwd() / REPO_NAME
    subprocess.check_call(['git', 'clone', REPO_URL, str(ROOT)])
    subprocess.check_call(['git', '-C', str(ROOT), 'checkout', REPO_REF])
sys.path.insert(0, str(ROOT / 'src'))
try:
    import numpy, scipy, pandas, matplotlib, torch
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(ROOT)])
from dynalign.experiments import environment, grid, run_grid
from dynalign.reporting import contrast_summary, plot_results
import pandas as pd
import matplotlib.pyplot as plt
print(environment())

## Choose the scale before running
`smoke` checks execution only; `pilot` explores settings; `full` uses five independent sensor/permutation worlds and two initialization/data repeats per world. These names do not guarantee adequate power. Keep the chosen plan fixed while collecting results.

Local and scrambled conditions have the same instantaneous cross-modal observation law, transition entropy, and transition spectrum. They differ in transition locality relative to the unchanged sensors. The IID condition is not entropy matched.

An update budget is held fixed across conditions and sample sizes. This changes data reuse as the dataset grows. Data pairs are independent length-two trajectories, not overlapping windows from one long sequence.

For persistence across VM termination, point `OUTPUT_DIR` at mounted durable storage. On Kaggle save a version and reattach its output on a new session. Checkpoints save progress every 100 updates and at completion.

In [ ]:
PROFILE = os.environ.get('DYNALIGN_PROFILE', 'smoke')
DEVICE = os.environ.get('DYNALIGN_DEVICE', 'auto')
base = Path('/kaggle/working') if Path('/kaggle/working').exists() else ROOT
OUTPUT_DIR = Path(os.environ.get('DYNALIGN_OUTPUT_DIR', str(base / 'outputs'))) / '02_sample_efficiency'
specs = grid('sample_efficiency', PROFILE)
display(pd.DataFrame(specs))
print(f'{len(specs)} condition runs; {2 * len(specs)} independently trained encoders')

## Train and evaluate
Each encoder minimizes a temporal VICReg-style objective: adjacent observations are encouraged to agree, with variance and covariance penalties against collapse. There is no shared-state target or cross-modal training loss. This practical objective is not identical to a hard-whitened spectral oracle.

A completed matching run is reused; an interrupted matching run resumes its checkpoint. Cache keys include source code, configuration, dependency versions, and device. The first run provides a runtime estimate; GPU-hours cannot be predicted from model size alone.

In [ ]:
results = run_grid(specs, OUTPUT_DIR, DEVICE)
display(results[['condition', 'n_pairs', 'world', 'repeat', 'initial_cka', 'cka', 'cka_gain', 'procrustes_error', 'any_collapse', 'training_seconds']])
print('Measured aggregate encoder-training seconds:', results['training_seconds'].sum())

## Compare conditions without selecting favorable runs
The paired contrast is local minus scrambled within the same world, sample budget, and initialization/data repeat. Repeats are averaged within world before world-level bootstrap intervals. With fewer than five valid worlds, intervals are left undefined. Five-world intervals remain preliminary.

The table retains collapsed runs. Inspect variance, effective rank, raw metrics, random-initialization similarity, and shuffled-correspondence similarity together. A gap in circular-coordinate probing can arise because those coordinates favor the local topology; transition-specific spectral probes provide a complementary target.

In [ ]:
contrasts = contrast_summary(results, 'cka')
contrasts.to_csv(OUTPUT_DIR / 'paired_contrasts.csv', index=False)
display(contrasts)
display(results[['condition', 'n_pairs', 'world', 'effective_rank_a', 'effective_rank_b', 'variance_a', 'variance_b', 'any_collapse', 'coordinate_r2_a', 'coordinate_r2_b', 'spectral_r2_a', 'spectral_r2_b', 'shuffled_cka']])
figure = plot_results(results, 'cka')
figure.savefig(OUTPUT_DIR / 'alignment_vs_data.png', dpi=150)
plt.show()

## Save and interpret
A positive contrast would support a finite-budget advantage in this specific setup, not an unrestricted convergence theorem. A null result, reversal, or failure to learn is also informative; report training diagnostics and uncertainty rather than changing the claim after seeing the outcome. These evaluations use fresh sensor noise at the same finite state support, not unseen-state or real-world generalization.

The archive contains configurations, environment records, checkpoints, embeddings, and summaries. Download or copy it to persistent storage.

In [ ]:
import shutil
from IPython.display import FileLink
archive = shutil.make_archive(str(OUTPUT_DIR), 'zip', OUTPUT_DIR)
print('Saved archive:', archive)
display(FileLink(archive))